# ViSta — a worked example, one step at a time

ViSta stacks interferometric observations of many faint galaxies **in the uv
plane** (on the visibilities, before any image is made) and measures the
stacked line and continuum.

This notebook shows how to use it, **one step at a time**, on observations
that are simulated, so that the right answer is known:

1. what the simulator makes, and one galaxy seen with and without noise;
2. a sample of twelve different galaxies, none of which shows the line alone;
3. the stack, and what it looks like in an image;
4. the measurement: how to read its numbers and its figures;
5. refining it: the window, the source model, the normalisations, the
   weighting, the errors, the continuum, each in its own step;
6. harder cases: a line that is not a Gaussian, sources that are rings, two
   lines in the band.

Every step changes **one thing**, says why, and shows what happens.

On Colab: *Runtime → Run all* (about 10 minutes on the CPU). Every number is
in the code cells: change it and run again from that cell.

## 0. Setup

The code is cloned from GitHub and its C++ kernel is built (a couple of
minutes, once). For a private repository, add a GitHub token to the Colab
secrets (key icon on the left) as `GITHUB_TOKEN`. Outside Colab, set
`VISTA_REPO` to a built checkout.

In [ ]:
REPO_URL = "https://github.com/martitors/ViSta-HPC.git"
BRANCH = "extract_opt"

import os, sys, glob, subprocess, logging
IN_COLAB = "google.colab" in sys.modules
REPO = os.environ.get("VISTA_REPO")            # a local checkout, to run this outside Colab
if REPO is None:
    REPO = "/content/ViSta-HPC" if IN_COLAB else os.path.abspath("ViSta-HPC")
    if not os.path.isdir(REPO):
        url = REPO_URL
        if IN_COLAB:
            try:                                 # private repository: token from the Colab secrets
                from google.colab import userdata
                url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
            except Exception:
                pass
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, REPO], check=True)
    else:                                        # a clone of an earlier session: update it
        head = lambda: subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"],
                                      capture_output=True, text=True).stdout
        before = head()
        subprocess.run(["git", "-C", REPO, "pull", "-q", "origin", BRANCH])
        if head() != before:                     # new code: build the kernel again, and
            for so in glob.glob(f"{REPO}/vista/ms_ops*.so"):     # forget what was imported
                os.remove(so)
            for name in [m for m in sys.modules if m.split(".")[0] in ("vista", "simulate")]:
                del sys.modules[name]
            if "ms_ops" in sys.modules:
                print("The code was updated: after this cell, Runtime > Restart session, "
                      "then Run all")

if IN_COLAB:
    subprocess.run(f"pip install -q -r {REPO}/requirements.txt pybind11", shell=True, check=True)
if not glob.glob(f"{REPO}/vista/ms_ops*.so"):          # build the kernel once
    subprocess.run(f"cmake -S {REPO} -B {REPO}/build -DCMAKE_BUILD_TYPE=Release > /dev/null"
                   f" && cmake --build {REPO}/build -j 2 > /dev/null"
                   f" && cmake --install {REPO}/build > /dev/null", shell=True, check=True)
sys.path[:0] = [REPO, os.path.join(REPO, "examples")]

import numpy as np
import matplotlib.pyplot as plt
import simulate                                        # examples/simulate.py: the simulator
import vista
from vista import ViSta

logging.getLogger("daskms").setLevel(logging.ERROR)    # quiet about empty optional columns
WORK = "/content/vista_demo" if IN_COLAB else os.path.abspath("vista_demo")
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
print("ViSta", vista.__version__, "from", REPO, "· working in", WORK)

## 1. The simulator, and one galaxy

`examples/simulate.py` writes Measurement Sets like those of a real
observation, so that ViSta reads them as it would read yours. For every
galaxy it makes:

* **the telescope**: an ALMA-like array of 43 antennas within `rmax` metres
  of its centre, observing for 20 minutes; the baselines rotate with the
  Earth, and the longer they are the finer the details they see;
* **the line**: a Gaussian in velocity, with integrated flux `line_flux`
  [Jy km/s] and width `fwhm_kms`, at the frequency where the redshift `z`
  puts it;
* **the continuum**: `cont_flux` [Jy], the dust emission under the line;
* **the shape**: a round Gaussian `size_arcsec` across (FWHM). An extended
  source is fainter on the long baselines;
* **the position**: `offset_arcsec` from the centre of the field, where the
  antennas are less sensitive (the *primary beam*);
* **the noise**: `noise` [Jy] on every visibility, correlated between
  adjacent channels as in real data. The weights written in the file
  underestimate it by a factor `kappa`, as real weights often do.

Here is one galaxy, written twice: without noise and with it.

In [ ]:
LINE = 345.79599                       # rest frequency of CO(3-2), GHz

galaxy = dict(z=2.5, rest_freq_ghz=LINE,
              line_flux=0.4, fwhm_kms=300,       # the line: 0.4 Jy km/s, 300 km/s wide
              cont_flux=2e-4,                    # the continuum: 0.2 mJy
              size_arcsec=0.8,                   # the shape
              offset_arcsec=(6, 3),              # 6" east and 3" north of the field centre
              noise=0.1, rmax=600)               # the noise and the size of the array

clean = simulate.make_ms("one_clean.ms", noiseless=True, **galaxy)     # without noise
noisy = simulate.make_ms("one_noisy.ms", **galaxy)                     # as observed

`simulate.plot_galaxy` shows them side by side. **Top**: the image of the
channel at the centre of the line (a *dirty* image: the Fourier transform of
the visibilities as they are; the circle is the resolution). **Bottom**: the
spectrum at the position of the galaxy (black), the average of all the
baselines.

Two green curves give the true signal. The **solid** one is what these
baselines should see: without noise the black spectrum lies exactly on it.
The **dashed** one is the total flux of the galaxy, higher: the source is
extended (0.8″), and the long baselines see only part of it. The total flux
is what a baseline of zero length would measure, and what ViSta estimates by
fitting how the flux falls with the baseline length (§4).

With the noise of a real observation neither the image nor the spectrum
show the galaxy clearly: this is the galaxy that stacking is for.

In [ ]:
simulate.plot_galaxy(clean, noisy)

## 2. A sample of twelve galaxies

`simulate.make_sample` writes one such observation per galaxy and the *input
list* that ViSta reads. The sample is heterogeneous on purpose, as a real
one:

* redshifts between 2 and 3, so the line falls at a different frequency in
  every dataset;
* line fluxes that scatter around 0.4 Jy km/s, widths between 250 and
  350 km/s, centres off by ±20 km/s (redshift errors), sizes between 0.4″
  and 1.2″;
* observations of different depth (`depth_spread_dex`) and with arrays of
  different size (`rmax`): different noise and different resolution;
* galaxies anywhere within 20″ of the field centre.

In [ ]:
truth = simulate.make_sample(
    "sample", n_sources=12, z_range=(2.0, 3.0), rest_freq_ghz=LINE,
    line_flux=0.4, flux_scatter_dex=0.15,                      # fluxes around 0.4 Jy km/s
    fwhm_kms=lambda rng: rng.uniform(250, 350),                # a different width each
    v_jitter_kms=20,                                           # redshift errors
    size_arcsec=lambda rng: rng.uniform(0.4, 1.2),             # a different size each
    cont_flux=2e-4,                                            # continuum, Jy
    noise=0.1, kappa=1.3, depth_spread_dex=0.3,                # noise; wrong weights; depths
    rmax=lambda rng: float(rng.choice([300, 600, 1200])),      # a different array each
    offset_arcsec=lambda rng: tuple(rng.uniform(-15, 15, 2)),  # position in the field
    seed=2)

print(simulate.describe(truth))

The input list has one line per dataset: the Measurement Set, the redshift
and the position of the galaxy. It is all ViSta needs to know about the
sample.

In [ ]:
print(open(truth["input_list"]).read())

What every observation sees on its own: the spectrum at the position of the
galaxy (black) and the true line (green). The deepest observations hint at
the line; in most it is lost in the noise, and in none can it be measured
well.

In [ ]:
simulate.plot_sample(truth)

**The expected answer.** Because the sample is simulated, we know what a
perfect measurement of the stack would give: `simulate.expected(truth)`
computes it from the true values (the mean of the galaxies, weighted as the
stack weights them). It is drawn in green in the comparisons below, as
`expected`. With real data there is no such thing: it is here only to check
the measurements.

In [ ]:
expected = simulate.expected(truth)
print(f"expected line flux of the stack: {expected['flux'][0]:.3f} Jy km/s")
print(f"expected continuum:              {expected['continuum'] * 1e3:.3f} mJy")

## 3. The stack

Two calls describe it:

* `ViSta(...)`: the input list and the name of what is written;
* `v.stack(...)`: the rest frequency of the line (`line_freq`, GHz, or a
  string as `"345.796GHz"`), the window that holds the line
  (`line_window=500`: from −500 to +500 km/s around it) and the band to keep
  (`velocity_range=900`: ±900 km/s).

`v.run()` then, for every dataset: brings the frequencies to the rest frame
of the galaxy, moves the phase centre onto the galaxy, puts the channels on
a common grid, and fits and subtracts the continuum on the channels outside
the line window. The result is one dataset, `stack.zarr`, with the
visibilities of all the galaxies.

In [ ]:
v = ViSta(truth["input_list"], output_name="stack")
v.stack(line_freq=LINE, line_window=500, velocity_range=900)
v.run()

**What stacking does, in an image.** `vista.dirty_image` makes the image of
the channels between two velocities; `sources=[5]` keeps only one galaxy of
the stack (the sixth, of average depth). On its own it shows at most a hint,
no stronger than the noise peaks around it; the twelve together show the
line at the centre, where every galaxy was moved.

(These images are a quick look. The measurements below are made on the
visibilities, not on images.)

In [ ]:
one = vista.dirty_image("stack.zarr", line_freq=LINE, velocity=(-150, 150), sources=[5],
                        fov_arcsec=16)
all12 = vista.dirty_image("stack.zarr", line_freq=LINE, velocity=(-150, 150), fov_arcsec=16)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
vista.plot_image(one, axes[0], title="one galaxy, −150 to +150 km/s")
vista.plot_image(all12, axes[1], title="the stack of twelve, −150 to +150 km/s")
fig.tight_layout()

## 4. The measurement

`v.extract()` asks for the measurement, here with all its defaults;
`v.run(only="measure")` makes it on the stack that is already there. It

1. compresses the stack into sums per channel and per *annulus* of baseline
   length (once: every later measurement reads only these sums and takes
   seconds);
2. brings every galaxy to the same redshift (`z_ref`, the median) and
   corrects it for the primary beam;
3. fits all channels and all annuli at once with a line that is a Gaussian
   in velocity, from a source that is a point or a Gaussian on the sky.

Printing the result gives what was measured **and the choices made**.

In [ ]:
v.extract()
r = v.run(only="measure")
print(r)

How to read it:

* **Line**: how sure the detection is. The centre and the width were
  searched, so the significance is lower than the S/N: it accounts for the
  search.
* **flux**: the number to quote, and where it comes from. Two fluxes are
  always measured: the **area of the Gaussian fit**, and the **area under
  the spectrum in the window** (a plain sum of the channels). The Gaussian
  one is quoted unless the two disagree, which would mean that the line is
  not a Gaussian (§12).
* **Gaussian**: centre, width and peak of the fitted line.
* **source**: the model of the sky that was chosen and why. A Gaussian
  source is used only if it fits better than a point source by a clear
  margin (Δχ² ≥ 9, i.e. 3σ).
* **Continuum**: what was subtracted (measured on its own) and whether
  anything is left outside the line window.
* **Reference**: the weighting and the redshift the fluxes refer to.

**The spectrum.** `r.plot("profile")` draws the stacked spectrum (black)
with its noise (grey band), the fitted Gaussian (red, its area hatched), the
window where the spectrum is summed (shaded) and the continuum level left
(dashed). The box repeats the two fluxes and says which is which. Below, the
residuals and the imaginary part (crosses) in units of the noise: both must
scatter around zero.

In [ ]:
r.plot("profile")

**The source.** `r.plot("uv")` draws the line flux measured in every annulus
of baseline length. A point source would give the same flux on all of them;
here it falls on the long baselines: the galaxies are extended. The model
(the thick curve) follows the points and its value on the left, at zero
baseline, is the total flux.

In [ ]:
r.plot("uv")

## 5. Refining: the window

The window of ±500 km/s is much wider than the line (about 300 km/s FWHM, so
all of it within ±300): the area flux sums 400 km/s of channels that hold
only noise, and its error is larger than it needs to be.

`window` sets the window of the area flux. `v.extract(...)` replaces all the
options of the measurement, and `run(only="measure")` repeats it in seconds.
The Gaussian fit does not change; the area flux now has a smaller error, and
the two fluxes are closer.

In [ ]:
v.extract(window=320)                  # the area flux: from -320 to +320 km/s
r = v.run(only="measure")
print(r)

In [ ]:
r.plot("profile")

A window too narrow would cut the line: the area flux would then be low,
while the Gaussian one would not move. The three windows side by side
(`vista.compare`: the dotted lines are the windows, the triangles the area
fluxes, the squares the Gaussian ones):

In [ ]:
v.extract(window=500)
wide = v.run(only="measure", tag="wide")         # tag: its files do not replace the others
v.extract(window=120)
narrow = v.run(only="measure", tag="narrow")

vista.compare({"window ±500": wide, "window ±320": r, "window ±120": narrow}, expected=expected)

(`line_window` of `v.stack` is another thing: the channels left out of the
continuum fit. It should hold the whole line too; to change it on the stack
that exists, `v.stack(..., line_window=...)` and
`v.run(only=["continuum", "measure"])`.)

From here on the window stays at ±320 km/s: `base` holds the options kept in
every measurement.

In [ ]:
base = dict(window=320)

## 6. The source model: the wrong one and the right one

What if we assume that the stacked source is a point? `model="point"` fits
the same flux on every baseline. The uv figure shows why it is wrong here:
the points fall with the baseline and the flat line cannot follow them. It
settles on an average, below the short baselines, and the flux is
underestimated.

In [ ]:
v.extract(**base, model="point")
point = v.run(only="measure", tag="point")
point.plot("uv")

The default, `model="auto"`, fits both and keeps the Gaussian only if the
data ask for it. Side by side (in the bottom-left panel the two models; on
the right the fluxes against the expected one):

In [ ]:
vista.compare({"model='auto'": r, "model='point'": point}, expected=expected)

When the size is known from elsewhere (the continuum, another line), it can
be given: `model=0.8` fixes it at 0.8″, `model_prior=(0.8, 0.15)` ties it
softly, `theta_range=(0.3, 2)` keeps it within limits. A fixed size has the
smallest error on the flux, if the size is right.

In [ ]:
v.extract(**base, model=0.8)
fixed = v.run(only="measure", tag="fixed")

vista.compare({"size fitted": r, "size fixed at 0.8\"": fixed}, expected=expected)

## 7. The normalisations

Before they are combined, the galaxies are made comparable in two ways. Each
can be switched off to see what it does.

**The redshift** (`z_norm`). The same galaxy is fainter if it is farther.
With `z_norm=True` every galaxy is brought to the flux it would have at
`z_ref`, so the stack is the mean of the same quantity. Without it the stack
is the mean of the fluxes as observed, at different distances. Here the
number is similar, because `z_ref` is the median redshift of the sample; but
it refers to no redshift, it changes with the redshifts of the galaxies that
happen to be in the sample, and the galaxies differ more from one another
(a larger error).

In [ ]:
v.extract(**base, z_norm=False)
no_z = v.run(only="measure", tag="no_z")

vista.compare({"z_norm=True (default)": r, "z_norm=False": no_z}, expected=expected)

**The primary beam** (`pb_corr`). An antenna is less sensitive away from the
centre of its field: these galaxies, up to 20″ away, are seen at 70–100 % of
their flux. With `pb_corr=True` every galaxy is corrected for the response
at its position. Without it the stack is low.

In [ ]:
v.extract(**base, pb_corr=False)
no_pb = v.run(only="measure", tag="no_pb")

vista.compare({"pb_corr=True (default)": r, "pb_corr=False": no_pb}, expected=expected)

## 8. The weighting

`weighting_scheme="natural"` (default) weights every visibility by its
noise: the deepest observations dominate, and the noise of the stack is the
lowest possible. `"democratic"` gives every galaxy the same total weight:
the stack is the plain mean of the galaxies, with more noise.

They answer two different questions, so they also have two different
expected values when the galaxies differ, as here.

In [ ]:
expected_democratic = simulate.expected(truth, weighting="democratic")
print(f"expected, natural:    {expected['flux'][0]:.3f} Jy km/s")
print(f"expected, democratic: {expected_democratic['flux'][0]:.3f} Jy km/s")

v.extract(**base, weighting_scheme="democratic")
democratic = v.run(only="measure", tag="democratic")

vista.compare({"natural": r, "democratic": democratic}, expected=expected)

## 9. The errors

The noise is not taken from the weights of the files: it is measured from
the scatter of the visibilities in every channel and annulus, and so is the
correlation between adjacent channels. `r.plot("noise")` shows the checks:

* left: the noise per channel from the scatter and from the weights. Here
  the weights are 30 % too optimistic (we simulated them so);
* centre: the stacked values away from the line, divided by their noise:
  they follow the bell curve if the noise is right;
* right: the same search for a line repeated on pure noise (grey), and the
  S/N measured (red line): how unlikely it is to be noise.

In [ ]:
r.plot("noise")

Trusting the weights instead (`noise="weights"`) gives the same flux, but an
error that is too small and a significance that is too high:

In [ ]:
v.extract(**base, noise="weights")
from_weights = v.run(only="measure", tag="weights")

vista.compare({"noise from the data (default)": r, "noise from the weights": from_weights},
              expected=expected)

## 10. The continuum

The galaxies have a continuum of about 0.2 mJy under the line. By default
(`continuum="subtract"` in `v.stack`) it is fitted on the channels outside
the line window and subtracted while stacking, then measured on its own:

In [ ]:
r.plot("continuum")

**Left in the data.** `continuum="none"` is for data where it was already
subtracted, or that have none. Here there is one: the line sits on it, the
fit takes it for part of the line and the flux is too high. ViSta notices:
it measures the level outside the window and warns. (The stack keeps the
data before the subtraction, so no new stack is needed: `run(only="measure")`
reads them again.)

In [ ]:
v.stack(line_freq=LINE, line_window=500, velocity_range=900, continuum="none")
v.extract(**base)
left = v.run(only="measure", tag="left")
left.plot("profile")

**Fitted with the line.** `continuum="fit"` leaves it in the data and fits
it together with the line, as a constant level: the dashed line of the
figure, with its value.

In [ ]:
v.stack(line_freq=LINE, line_window=500, velocity_range=900, continuum="fit")
fitted = v.run(only="measure", tag="fitted")
fitted.plot("profile")

The three side by side. Subtracting and fitting agree; leaving the
continuum in does not.

In [ ]:
v.stack(line_freq=LINE, line_window=500, velocity_range=900)     # back to the subtraction

vista.compare({"subtracted (default)": r, "fitted with the line": fitted,
               "left in the data": left}, expected=expected)

## 11. A harder case: a line that is not a Gaussian

A second sample: the line of every galaxy has broad wings (a third of the
flux in a component 800 km/s wide, as an outflow would give), and the
observations are deeper. The band is wider, to hold the wings.

In [ ]:
truth_wings = simulate.make_sample(
    "wings", n_sources=12, z_range=(2.0, 3.0), rest_freq_ghz=LINE,
    line_flux=0.4, fwhm_kms=250, size_arcsec=0.5, noise=0.05, rmax=600, seed=1,
    nchan=150,                                                             # a wider band
    extra_lines=[dict(rest_freq_ghz=LINE, flux_ratio=0.5, fwhm_kms=800)])  # the wings
expected_wings = simulate.expected(truth_wings)

w = ViSta(truth_wings["input_list"], output_name="wings")
w.stack(line_freq=LINE, line_window=1000, velocity_range=1400)
w.extract()
wings = w.run()                        # stack and measure, in one go
print(wings)

The Gaussian fit (red) follows the core of the line and misses the wings:
its area is too small. The sum of the spectrum in the window has no shape to
assume and catches them. ViSta sees that the two disagree by more than their
errors allow, and quotes the area: the report above says so, and why.

In [ ]:
wings.plot("profile")

Forcing one or the other (`flux_measure="gauss"`, `"area"`) against the
expected flux:

In [ ]:
w.extract(flux_measure="gauss")
wings_gauss = w.run(only="measure", tag="gauss")

vista.compare({"flux_measure='auto'": wings, "flux_measure='gauss'": wings_gauss},
              expected=expected_wings)

## 12. A harder case: sources that are not Gaussians

A third sample: every galaxy is a **ring** 3″ across. ViSta models the
stacked source as a point or a round Gaussian: what happens when it is
neither?

In [ ]:
truth_ring = simulate.make_sample(
    "ring", n_sources=12, z_range=(2.0, 3.0), rest_freq_ghz=LINE,
    line_flux=0.6, fwhm_kms=300, size_arcsec=3.0, shape="ring",     # a ring 3" across
    noise=0.04, rmax=600, seed=5)

g = ViSta(truth_ring["input_list"], output_name="ring")
g.stack(line_freq=LINE, line_window=500, velocity_range=900)
g.extract(window=320)
ring = g.run()
ring.plot("image")

The image of the stack shows the ring. In the uv plane a ring has a
signature: its flux goes **negative** on some baselines, which no Gaussian
can do. The fitted Gaussian follows the short baselines, which set the total
flux, and misses the rest. The flux is still right within its error, because
it is decided at short baselines; the size is that of "a Gaussian as wide as
the ring", not a property of the galaxies.

The lesson for real data: look at the uv figure and at the image. Points
that go systematically below the model, or below zero, say that the source
is not what the model assumes.

In [ ]:
vista.compare({"Gaussian source model": ring}, expected=simulate.expected(truth_ring))

## 13. A harder case: two lines in the band

A fourth sample: every galaxy has a second line, half as bright, 400 km/s
from the first. If only the first is listed, part of the second falls in its
window and part where the continuum is fitted: the fit has one Gaussian for
two lines.

In [ ]:
LINE2 = LINE * (1 - 400 / 299792.458)          # 400 km/s from the first
truth_two = simulate.make_sample(
    "two", n_sources=12, z_range=(2.0, 3.0), rest_freq_ghz=LINE,
    line_flux=0.4, fwhm_kms=250, size_arcsec=0.5, noise=0.05, rmax=600, seed=2, nchan=150,
    extra_lines=[dict(rest_freq_ghz=LINE2, flux_ratio=0.5, fwhm_kms=250)])

d = ViSta(truth_two["input_list"], output_name="two")
d.stack(line_freq=LINE, line_window=300, velocity_range=1400)      # only the first line
d.extract()
first_only = d.run()
first_only.plot("profile")

Listing both (`line_freq=[LINE, LINE2]`, each with its window) makes a new
stack, because the channels kept out of the continuum fit change. The
windows overlap, so the two lines are fitted **together**, one Gaussian
each, and separated. (Lines whose windows do not overlap are measured each
on its own.)

In [ ]:
d.stack(line_freq=[LINE, LINE2], line_window=300, velocity_range=1400)
both = d.run()
print(both)

In [ ]:
both.plot("profile")

## 14. The command line

`python -m vista` (or `run_vista.sh`) has the same options as `ViSta()`,
`stack()` and `extract()`, one flag each; `--extract` asks for the
measurement. It writes the same files: the stack, `*_results.json` with
every number, and the figures as PNG.

In [ ]:
!PYTHONPATH={REPO} python -m vista --input sample/input_list.txt --output-name cli/stack \
    --line-freq 345.796GHz --line-window 500 --velocity-range 900 \
    --extract --window 320

## 15. On your own data

1. Write the input list, one line per dataset:
   `<ms_path> <redshift> <RA> <Dec> [FIELD_ID] [SPW_IDS] [NORM]`, with the
   position of the galaxy (hh:mm:ss.s ±dd:mm:ss.s).
2. `python examples/inspect_ms.py input_list.txt --line-freq 345.796GHz --all`
   reads the bands of the datasets and suggests `velocity_range`,
   `line_window` and `annuli`.
3. Stack and measure as in §3 and §4, with a window wide enough for the
   whole line.
4. Read the report (`print(r)`) and look at `r.plot("profile")`,
   `r.plot("uv")`, `r.plot("noise")`, `r.plot("image")`: the residuals and
   the imaginary part around zero, nothing left of the continuum, the model
   on the points.
5. Refine as in §5–§10, one option at a time, with `run(only="measure")`.
   `v.help()` lists every option with its value and what it can be.

The documentation, <https://vista-hpc.readthedocs.io>, explains every option.